# Autoresearch Experiment Analysis

Analysis of autonomous hyperparameter tuning results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 5 columns: commit, val_avg_attack_recall, memory_gb, status, description)
df = pd.read_csv('results.tsv', sep='\t')
df['val_avg_attack_recall'] = pd.to_numeric(df['val_avg_attack_recall'], errors='coerce')
df['memory_gb'] = pd.to_numeric(df['memory_gb'], errors='coerce')
df['status'] = df['status'].str.strip().str.upper()

print(f'Total experiments: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head(10)

In [ ]:
counts = df['status'].value_counts()
print('Experiment outcomes:')
print(counts.to_string())

n_keep = counts.get('KEEP', 0)
n_discard = counts.get('DISCARD', 0)
n_crash = counts.get('CRASH', 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f'\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}')

In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
print(f'KEPT experiments: {len(kept)}')
kept[['commit', 'val_avg_attack_recall', 'memory_gb', 'description']]

## Val avg_attack_recall Over Time

Track how the best (kept) recall evolves as experiments progress. Higher is better.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

valid = df[df['status'] != 'CRASH'].copy()
valid['exp_idx'] = range(len(valid))

# All non-crash experiments
ax.scatter(valid['exp_idx'], valid['val_avg_attack_recall'],
           c=valid['status'].map({'KEEP': 'green', 'DISCARD': 'red'}),
           alpha=0.6, s=40, zorder=3, label='experiments')

# Running best (cumulative max)
running_best = valid['val_avg_attack_recall'].cummax()
ax.plot(valid['exp_idx'], running_best, 'b-', linewidth=2, label='running best')

ax.set_xlabel('Experiment index')
ax.set_ylabel('val_avg_attack_recall')
ax.set_title('Autoresearch progress — FlowGuard IDS')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('progress.png', dpi=150)
plt.show()

## Summary Statistics

In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
baseline_recall = df.iloc[0]['val_avg_attack_recall']
best_recall = kept['val_avg_attack_recall'].max() if len(kept) else baseline_recall
improvement = best_recall - baseline_recall

print(f'Baseline recall:  {baseline_recall:.6f}')
print(f'Best recall:      {best_recall:.6f}')
print(f'Improvement:      +{improvement:.6f}')
print(f'Memory (avg kept): {kept["memory_gb"].mean():.1f} GB' if len(kept) else 'No kept experiments yet')

## Top Improvements (Kept Experiments)

In [ ]:
kept = df[df['status'] == 'KEEP'].copy().reset_index(drop=True)
# Delta vs previous kept experiment
kept['delta'] = kept['val_avg_attack_recall'].diff().fillna(
    kept['val_avg_attack_recall'].iloc[0] - df.iloc[0]['val_avg_attack_recall']
    if len(kept) else 0
)
top = kept.sort_values('delta', ascending=False).head(10)
top[['commit', 'val_avg_attack_recall', 'delta', 'memory_gb', 'description']]